# Produzione del TTree per l'analisi dati

- Informazione temporale:
        delta_t_scope = 0.025 ns
        t_min = -25.0 ns
        t_max = +25.4 ns


- Branches del Tree scope:

        - scope_tree.Branch("baseline",         "baseline[4][3]/D")     
                Baseline  (Nevento; Pixel; (valore,rms,err))

        - scope_tree.Branch("underline",        "underline[4][3]/D")
                Underline (Nevento; Pixel; (valore,rms,err))

        - scope_tree.Branch("signal",           "signal[4]/D")          
                Signal    (Nevento; Pixel)

        - scope_tree.Branch("t",                "t[4][10]/D")           
                CFD_times (Nevento; Pixel; (t0,t10,...,t90))

- Branches del Tree ADC:

        - adc_tree.Branch("baseline",           "baseline[4][4]/D")
                Baseline        # (Nevento; Riga; Colonna)

        - adc_tree.Branch("rms",                "rms[4][4]/D")
                RMS             # (Nevento; Riga; Colonna)

        - adc_tree.Branch("p2p",                "p2p[4][4]/D")
                P2P             # (Nevento; Riga; Colonna)

        - adc_tree.Branch("data_centered",      "data_centered[4][4][200]/D")
                Data_centered   # (Nevento; Riga; Colonna; ????) =================

        - adc_tree.Branch("signal",             "signal[4][4]/D")
                Signal          # (Nevento; Riga; Colonna)

        - adc_tree.Branch("frame_max",          "frame_max/D")
                Frame_max       # (Nevento)

        - adc_tree.Branch("noise",              "noise[4][4]/D")
                Noise           # (Nevento; Riga; Colonna)



In [6]:
import pickle
import numpy as np
import ROOT

filename = "not_calibrated/signal_extraction_return_dictionary_run433171958_261021172134.pkl"

with open(filename, "rb") as file:
    data = pickle.load(file)


# Informazione temporale oscilloscopio
delta_t_scope = 0.025   #ns
t_min = -25.            #ns
t_max = 25.4            #ns


# Strutture dati oscilloscopio
scope_baseline_struct  = np.asarray(data['scope']['baseline'], dtype=np.float64)    # (Nevento; Pixel; (valore,rms,err))
scope_underline_struct = np.asarray(data['scope']['underline'], dtype=np.float64)   # (Nevento; Pixel; (valore,rms,err))
scope_CFD_times_struct = np.asarray(data['scope']['CFD_times'], dtype=np.float64)   # (Nevento; Pixel; (t0,10,t20,...,t90))
scope_signal_struct    = np.asarray(data['scope']['signal'], dtype=np.float64)      # (Nevento; Pixel)


# Strutture dati ADC
adc_baseline_struct    = np.asarray(data['ADC']['baseline'], dtype=np.float64)      # (Nevento; Riga; Colonna)
adc_rms_struct         = np.asarray(data['ADC']['rms'], dtype=np.float64)           # (Nevento; Riga; Colonna)
adc_p2p_struct         = np.asarray(data['ADC']['p2p'], dtype=np.float64)           # (Nevento; Riga; Colonna)
adc_centered_struct    = np.asarray(data['ADC']['data_centered'], dtype=np.float64) # (Nevento; Riga; Colonna; ????) =================
adc_signal_struct      = np.asarray(data['ADC']['signal'], dtype=np.float64)        # (Nevento; Riga; Colonna)
adc_frame_max_struct   = np.asarray(data['ADC']['frame_max'], dtype=np.float64)     # (Nevento)
adc_noise_struct       = np.asarray(data['ADC']['noise'], dtype=np.float64)         # (Nevento; Riga; Colonna)
#nota: ADC_centered_struct è un array 10001 x 4 x 4 x 200, capire cosa sia l'ultimo valore


#Creazione file root con TREE contenente ampiezze e risetime = t50-t10 per ogni pixel dello scope, per ogni evento
outfile = "data.root"
b_signal = np.zeros(4, dtype=np.float64)
b_t      = np.zeros((4, 10), dtype=np.float64)
b_base   = np.zeros((4, 3), dtype=np.float64)
b_under  = np.zeros((4, 3), dtype=np.float64)

a_baseline = np.zeros((4, 4), dtype=np.float64)
a_rms      = np.zeros((4, 4), dtype=np.float64)
a_p2p      = np.zeros((4, 4), dtype=np.float64)
a_centered = np.zeros((4, 4, 200), dtype=np.float64)
a_signal   = np.zeros((4, 4), dtype=np.float64)
a_frame_max = np.zeros(1, dtype=np.float64)
a_noise    = np.zeros((4, 4), dtype=np.float64)

fout = ROOT.TFile(outfile, "RECREATE")
scope_tree = ROOT.TTree("Scope data", "Scope data")
adc_tree = ROOT.TTree("ADC data", "ADC data")

scope_tree.Branch("baseline",    b_base, "baseline[4][3]/D")
scope_tree.Branch("underline",   b_under, "underline[4][3]/D")
scope_tree.Branch("signal",      b_signal, "signal[4]/D")
scope_tree.Branch("t",           b_t,      "t[4][10]/D")

adc_tree.Branch("baseline",      a_baseline, "baseline[4][4]/D")
adc_tree.Branch("rms",           a_rms,      "rms[4][4]/D")
adc_tree.Branch("p2p",           a_p2p,      "p2p[4][4]/D")
adc_tree.Branch("data_centered", a_centered, "data_centered[4][4][200]/D")
adc_tree.Branch("signal",        a_signal,   "signal[4][4]/D")
adc_tree.Branch("frame_max",     a_frame_max, "frame_max/D")
adc_tree.Branch("noise",         a_noise,    "noise[4][4]/D")

for i in range(len(adc_frame_max_struct)):
    b_base[:, :] = scope_baseline_struct[i]
    b_under[:, :] = scope_underline_struct[i]
    b_signal[:] = scope_signal_struct[i]
    b_t[:, :]   = scope_CFD_times_struct[i]
    scope_tree.Fill()

    a_baseline[:, :] = adc_baseline_struct[i]
    a_rms[:, :]      = adc_rms_struct[i]
    a_p2p[:, :]      = adc_p2p_struct[i]
    a_centered[:, :, :] = adc_centered_struct[i]
    a_signal[:, :]   = adc_signal_struct[i]
    a_frame_max[:]   = adc_frame_max_struct[i]
    a_noise[:, :]    = adc_noise_struct[i]
    adc_tree.Fill()

scope_tree.Write()
adc_tree.Write()
fout.Close()

/var/folders/_c/1lttd7hj1xlb8px4wb2_yg4m0000gn/T/ipykernel_24079/2742421013.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(file)


In [ ]:
#pixel mapping
#                       | 0| 1| 2| 3|
#                       | 4| 5| 6| 7|
#                       | 8| 9|10|11|
#                       |12|13|14|15| 
